## Joint Ocean Ice Study

The JOIS datasets comprise four years (2021-2024) of seawater radionuclide measurements from the Beaufort Sea, collected as part of the Joint Ocean Ice Study (JOIS) expeditions aboard the CCGS Louis St. Laurent. Samples were analysed at ETH Zurich / LIP (Nuria Casacuberta's group). The raw data are published as [Zenodo archives](https://zenodo.org/communities/titanica/records?q=&l=list&p=1&s=10&sort=newest), each containing a single Excel file with CTD (conductivity, temperature, depth) metadata and activity concentrations in wide format.

Nuclide coverage varies by year:

:::{style="width: fit-content; font-size: 0.85em; margin: 0 auto;"}
| Year | I-129 | U-236 | U-238 | U-236/U-238 |
|------|:-----:|:-----:|:-----:|:-----------:|
| 2021 | ✓ | | | |
| 2022 | ✓ | ✓ | ✓ | ✓ |
| 2023 | ✓ | ✓ | ✓ | ✓ |
| 2024 | ✓ | | | |
:::

The column layout is similar enough across years that the same pipeline handles all of them, with a few normalisation steps to absorb the differences.

In [ ]:
#| default_exp handlers.jois

In [ ]:
#| export
from fastcore.all import *
import pandas as pd
import numpy as np
import re
import requests
import zipfile
import io

from marisco.callbacks import (PerGroupCB, Callback, Transformer, EncodeTimeCB,
                                SanitizeLonLatCB, RemapCB, AddSampleIDCB)
from marisco.metadata import GlobAttrsFeeder, ZoteroCB, BboxCB, DepthRangeCB, TimeRangeCB, KeyValuePairCB
from marisco.encoders import NetCDFEncoder

In [ ]:
#| exports
RECORDS = {
    2021: {'url': 'https://zenodo.org/records/18880401/files/annabel-payne/BGOS-JOIS-2021-v1.0.1.zip?download=1'},
    2022: {'url': 'https://zenodo.org/records/18880777/files/annabel-payne/BGOS-JOIS-2022-v1.0.1.zip?download=1'},
    2023: {'url': 'https://zenodo.org/records/18880591/files/annabel-payne/BGOS-JOIS-2023-v1.0.zip?download=1'},
    2024: {'url': 'https://zenodo.org/records/18880497/files/annabel-payne/BGOS-JOIS-2024-v1.0.1.zip?download=1'},
}
fname_out = 'JOIS_Beaufort_Sea.nc'
src_dir   = None  # remote-only, no local files

## Raw data format

All four JOIS ZIP archives contain a single Excel sheet with CTD metadata columns and activity concentrations in wide format: each nuclide-unit pair gets its own column (e.g. `I129_at_kg`, `I129_at_l`), and each value column has a matching `unc_` column for uncertainty. Value columns carry `( x 10^N)` suffixes in some years indicating an unscaled value. The 2021 ZIP has a stray `( x 10^6)` column header that should read `Cruise`.

`load_data` handles all four years by:
- Normalising column names (stripping scale-factor suffixes)
- Extracting and applying scale factors to unscaled values
- Detecting the 2023 U-236 `at_kg` columns that are missing their suffix but are in the same scale as their `at_l` counterparts
- Concatenating all years into a single DataFrame

:::{.callout-important}
## FEEDBACK TO DATA PROVIDER

Inconsistent scale-factor application across years: `I129_at_kg` values in the 2021 and 2022 Excel files are reported as raw x10^-7 (columns named `I129_at_kg ( x 10^7)`), while 2023-2024 use the same column name `I129_at_kg` with the scale already applied. The same issue affects `U236_at_kg` in 2023: `U236_at_l` has its `( x 10^6)` suffix, but `U236_at_kg` does not, even though both are in the same scale (ratio check confirms seawater density ~1025 kg/m3). The provider should confirm whether 2021-2022 data can be published with the scale applied.
:::

In [ ]:
#| export
def norm_cols(cols  # Column names to normalise
    ) -> list:
    "Normalise column names: strip scale-factor suffixes like ( x 10^7) or (x 10^6)."
    return [re.sub(r'\s*\([^)]*\)\s*', '', c).strip() for c in cols]

In [ ]:
test_eq(norm_cols(['I129_at_kg ( x 10^7)', 'U236_at_l']),
        ['I129_at_kg', 'U236_at_l'])
test_eq(norm_cols(['Stn', 'Depth_m']), ['Stn', 'Depth_m'])
print("norm_cols: no-change case and suffix-stripping case pass. ✓")

norm_cols: no-change case and suffix-stripping case pass. ✓


In [ ]:
def extract_scales(cols  # Column names to scan for scale-factor suffixes
    ) -> dict:
    "Return {normalized_col: factor} for columns with `( x 10^N)` suffix, supporting signed exponents."
    return {k: 10**int(m.group(1)) for c in cols
            if (m := re.search(r'\s*\(\s*x\s*10\^([+-]?\d+)\)', c))
            and (k := re.sub(r'\s*\([^)]*\)\s*', '', c).strip())}


In [ ]:
scales = extract_scales(['I129_at_kg ( x 10^7)', 'U236_at_l(x10^6)', 'Stn'])
test_eq(scales, {'I129_at_kg': 10_000_000, 'U236_at_l': 1_000_000})
print("extract_scales: two scales extracted, no spurious matches. ✓")

extract_scales: two scales extracted, no spurious matches. ✓


In [ ]:
def apply_scales(
    df,     # DataFrame to modify in place
    scales, # {col: factor} of scale factors to apply
    ) -> pd.DataFrame:
    "Multiply columns in df by their scale factor."
    for col, factor in scales.items():
        if col in df.columns: df[col] *= factor
    return df


In [ ]:
df = pd.DataFrame({'I129_at_kg': [1.0, 2.0], 'Stn': ['A', 'B']})
scales = {'I129_at_kg': 10_000_000}
result = apply_scales(df, scales)
test_eq(result['I129_at_kg'].tolist(), [10_000_000.0, 20_000_000.0])
test_eq(result['Stn'].tolist(), ['A', 'B'])
print("apply_scales: value columns scaled, non-value columns unchanged. ✓")

apply_scales: value columns scaled, non-value columns unchanged. ✓


In [ ]:
#| export
def load_data(
    recs=None, # Optional dict of year->record; defaults to all RECORDS
    ) -> dict:
    "Fetch all JOIS ZIP archives from Zenodo and return raw DataFrames with original column names intact."
    recs = recs or RECORDS
    parts = []
    for r in recs.values():
        resp = requests.get(r['url'], timeout=60); resp.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
            xl_name = next(n for n in z.namelist() if n.lower().endswith(('.xlsx', '.xls')))
            df = pd.ExcelFile(io.BytesIO(z.read(xl_name))).parse(sheet_name=0)
        cols = list(df.columns)
        if norm_cols(cols)[0] == '': cols[0] = 'Cruise'  # 2021 ZIP: stray empty header in col 0
        df.columns = cols
        parts.append(df)
    return {'SEAWATER': pd.concat(parts, ignore_index=True)}


In [ ]:
#| eval: false
dfs = load_data()

In [ ]:
#|eval: false
print(dfs['SEAWATER'].describe(include='number').T[['count', 'mean', 'min', 'max']])

                   count          mean          min           max
sample_number      441.0  6.377082e+02    25.000000  1.209000e+03
Latitude_degN      441.0  7.469298e+01    70.539500  7.969150e+01
Longitude_degE     441.0 -1.452820e+02  -153.320167 -1.229452e+02
Pressure_dbar      441.0  7.458816e+02     4.223000  3.817376e+03
Depth_m            441.0  7.350311e+02     4.179505  3.743998e+03
Temperature_degC   441.0 -1.417129e-01    -1.586100  5.142400e+00
Conservative_Temp  441.0 -1.736566e-01    -1.579121  5.199772e+00
Potential_Temp     441.0 -1.787647e-01    -1.586842  5.142001e+00
Salinity_psu       441.0  3.343552e+01    24.756100  3.495950e+01
Absolute_Salinity  441.0  3.359702e+01    24.874988  3.512872e+01
Sigma0             441.0  2.685045e+01    19.866195  2.810244e+01
Insitu_Density     441.0  1.026899e+00     1.019916  1.028151e+00
I129_at_kg         441.0  1.097553e+09     0.000000  6.033279e+09
unc_I129_at_kg     441.0  4.149385e+07     0.000000  2.503235e+08
unc_129_pc

## Rename and standardise columns

The raw JOIS columns encode nuclide, unit, and sometimes method in a single string.
We map CTD column names to MARIS uppercase standards, and combine separate date/time columns.

Two callbacks handle this. Both run on the `SEAWATER` group only.

`RenameColsCB` maps CTD columns to MARIS names. `ParseDateTimeCB` combines `Date` and `Time`.
The nuclide columns (`I129_at_kg`, `U238_ppb`, etc.) are left with their original names at this stage;
`MeltAndScaleCB` handles their identification, scale application, and reshape in one pass downstream.


:::{.callout-important}
## FEEDBACK TO DATA PROVIDER

Column headers in the JOIS and GEOTRACES datasets encode multiple pieces of information (nuclide, unit, measurement method) into a single string. This adds friction to data ingestion, since every new dataset with a different naming convention requires custom parsing logic. MARIS prefers a tidy data layout ([Wickham 2014, doi:10.18637/jss.v059.i10](https://www.jstatsoft.org/article/view/v059i10)) where each column holds a single variable, and metadata like nuclide, unit, and method are stored as separate columns, not baked into the header.
:::

In [ ]:
#| export
class RenameColsCB(PerGroupCB):
    "Map JOIS provider CTD and sample columns to MARIS standard names."
    grps = ["SEAWATER"]
    def each_grp(self, grp, df, tfm):
        df.rename(columns={
            "Latitude_degN": "LAT", "Longitude_degE": "LON",
            "Depth_m": "SMP_DEPTH", "Temperature_degC": "TEMP",
            "Salinity_psu": "SAL", "Station": "STATION",
            "sample_number": "SMP_ID_PROVIDER",
        }, inplace=True)

In [ ]:
# Verify RenameColsCB maps provider columns to MARIS names
dfs_mock = {'SEAWATER': pd.DataFrame({'Latitude_degN': [70.5], 'Longitude_degE': [-140.0],
                                       'Depth_m': [200.0], 'sample_number': [101]})}
tfm = Transformer(dfs_mock, cbs=[RenameColsCB()])
tfm()
test_eq('LAT' in tfm.dfs['SEAWATER'].columns, True)
test_eq('LON' in tfm.dfs['SEAWATER'].columns, True)
test_eq('SMP_DEPTH' in tfm.dfs['SEAWATER'].columns, True)
test_eq('SMP_ID_PROVIDER' in tfm.dfs['SEAWATER'].columns, True)
print("RenameColsCB: provider columns mapped to MARIS names. ✓")


RenameColsCB: provider columns mapped to MARIS names. ✓


In [ ]:
#| export
class ParseDateTimeCB(PerGroupCB):
    "Combine JOIS Date and Time columns into a single UTC-aware TIME column."
    grps = ["SEAWATER"]
    def __init__(self, col_date="Date",  # Source date column name
                 col_time="Time"):       # Source time column name
        store_attr()
    def each_grp(self, grp, df, tfm):
        df["TIME"] = pd.to_datetime(
            df[self.col_date].astype(str) + "T" + df[self.col_time].astype(str),
            utc=True)
        df.drop(columns=[self.col_date, self.col_time], inplace=True)


In [ ]:
# Verify ParseDateTimeCB combines Date and Time into UTC-aware TIME
dfs_mock = {'SEAWATER': pd.DataFrame({'Date': ['2021-08-19'], 'Time': ['08:00:00']})}
tfm = Transformer(dfs_mock, cbs=[ParseDateTimeCB()])
tfm()
test_eq('TIME' in tfm.dfs['SEAWATER'].columns, True)
test_eq('Date' not in tfm.dfs['SEAWATER'].columns, True)
test_eq(str(tfm.dfs['SEAWATER']['TIME'].dt.tz), 'UTC')
print(f"ParseDateTimeCB: TIME = {tfm.dfs['SEAWATER']['TIME'].iloc[0]}, tz=UTC. ✓")


ParseDateTimeCB: TIME = 2021-08-19 08:00:00. ✓


::: {.callout-note}

#### Note for MARIS DB team

The JOIS datasets include a `Pressure_dbar` column with CTD pressure values. This parameter is currently not present in the MARIS output schema. If useful for the database, it could be added alongside the other CTD metadata fields.
  The extra CTD columns (`Conservative_Temp`, `Potential_Temp`, `Absolute_Salinity`, `Sigma0`, `Insitu_Density`, `unc_129_pct`) and the `Cruise` column are not mapped in the NC_CSV dict (the central remapping from internal column names to NetCDF/CSV output names), so they are dropped after the melt. If the data team decides these fields are useful, the fix should go in NC_CSV, not in this handler.

:::

## Reshape wide to long

The raw JOIS data uses wide format with nuclide-unit concentrations spread across columns.
MARIS requires long format: one row per measurement with `NUCLIDE`, `UNIT`, `VALUE`, `UNC` columns.

`MeltAndScaleCB` performs all of the following in a single column-scan pass:

1. **Column identification** — `JOIS_COL_LUT` maps each normalized column name to `(nuc_tok, src_unit_tok)`,
   eliminating the need for a pre-rename shim and a hand-written `VAL_COLS` list.
2. **Scale-suffix extraction** — reads `( x 10^N)` from the raw column name before stripping it.
   Missing-suffix overrides are handled by `COL_SCALE_OVERRIDES` (dict lookup; zero `if` branches).
3. **Unit conversion** — `UNIT_CONV_LUT` maps source unit tokens to `(maris_unit_tok, base_factor)`;
   for `U238_ppb` this absorbs the ppb→atoms/kg conversion, replacing the former `ConvertU238CB`.
4. **VALUE/UNC pairing** — the matching `unc_{col}` column is located by name derivation;
   no merge on metadata keys, so duplicate-row Cartesian blowup is structurally impossible.


In [ ]:
#| exports
# Columns kept as identifiers during the wide-to-long reshape
META_COLS = ['Cruise', 'STATION', 'SMP_ID_PROVIDER', 'LAT', 'LON',
             'TIME', 'Pressure_dbar', 'SMP_DEPTH', 'TEMP', 'SAL']


In [ ]:
#| exports
# Convert U-238 from ppb to atoms/kg: ppb * 1e-9 * (1/238.05) * 6.02214076e23
U238_PPB_TO_AT_KG = 2.529_697e12

# Maps normalized column name -> (nuc_tok, src_unit_tok)
# Eliminates VAL_COLS enumeration + RenameNucColsCB naming shim + MeltJOISCB._at_ split
JOIS_COL_LUT = {
    'I129_at_kg':  ('I129',      'at_kg'),
    'I129_at_l':   ('I129',      'at_l'),
    'U236_at_kg':  ('U236',      'at_kg'),
    'U236_at_l':   ('U236',      'at_l'),
    'U238_ppb':    ('U238',      'ppb'),    # src unit; UNIT_CONV_LUT normalises to at_kg
    'U236_U238':   ('U236_U238', 'ratio'),  # src unit; UNIT_CONV_LUT normalises to at_ratio
}

# Maps src_unit_tok -> (maris_unit_tok, base_factor)
# Absorbs ConvertU238CB: ppb->at_kg factor lives here, not in a separate Callback
UNIT_CONV_LUT = {
    'at_kg':    ('at_kg',    1.0),
    'at_l':     ('at_l',     1.0),
    'at_ratio': ('at_ratio', 1.0),
    'ppb':      ('at_kg',    U238_PPB_TO_AT_KG),   # U238 ppb -> atoms/kg
    'ratio':    ('at_ratio', 1.0),
}

# Scale overrides for columns whose ( x 10^N) suffix is absent in the provider file.
# 2023 U236_at_kg: suffix absent; scale confirmed by at_l/at_kg density-ratio check (~1025 kg/m3).
# See FEEDBACK TO DATA PROVIDER callout above for full provenance.
COL_SCALE_OVERRIDES = {
    'U236_at_kg':     1e6,
    'unc_U236_at_kg': 1e6,
}


In [ ]:
#| export
class MeltNuclideColsCB(PerGroupCB):
    "Reshape JOIS nuclide columns from wide to long, scaling each value by its column-name suffix factor and unit-conversion factor in the same pass. COL_SCALE_OVERRIDES supplies factors for suffix-absent columns (e.g. 2023 U236_at_kg: factor 1e6 confirmed by at_l/at_kg density-ratio check, seawater density ~1025 kg/m3)."
    grps = ['SEAWATER']
    def __init__(self,
                 meta_cols,                  # Identifier columns to preserve
                 col_lut,                    # {normalized_col: (nuc_tok, src_unit_tok)}
                 unit_conv_lut,              # {src_unit_tok: (maris_unit_tok, base_factor)}
                 col_scale_overrides=None):  # {normalized_col: factor} for suffix-absent columns
        store_attr()
        self.col_scale_overrides = col_scale_overrides or {}
    def each_grp(self, grp, df, tfm):
        frames = []
        for col in list(df.columns):
            norm = re.sub(r'\s*\([^)]*\)\s*', '', col).strip()
            if norm not in self.col_lut or norm.startswith('unc_'): continue
            nuc_tok, src_unit_tok = self.col_lut[norm]
            m = re.search(r'\s*\(\s*x\s*10\^([+-]?\d+)\)', col)
            suffix_factor = 10**int(m.group(1)) if m else self.col_scale_overrides.get(norm, 1.0)
            maris_unit_tok, base_factor = self.unit_conv_lut[src_unit_tok]
            total_factor = suffix_factor * base_factor
            unc_raw = next((c for c in df.columns
                            if re.sub(r'\s*\([^)]*\)\s*', '', c).strip() == f'unc_{norm}'), None)
            sub = df[self.meta_cols + [col]].copy()
            sub.rename(columns={col: 'VALUE'}, inplace=True)
            sub['VALUE'] = sub['VALUE'] * total_factor
            sub['UNC'] = df[unc_raw].values * total_factor if unc_raw else np.nan
            sub['NUCLIDE'] = nuc_tok
            sub['UNIT'] = maris_unit_tok
            frames.append(sub)
        out = pd.concat(frames, ignore_index=True)
        out.dropna(subset=['VALUE'], inplace=True)
        tfm.dfs[grp] = out


In [ ]:
# Verify MeltNuclideColsCB: scale suffix, unit conversion, and COL_SCALE_OVERRIDES applied correctly
dfs_mock = {'SEAWATER': pd.DataFrame({
    'Cruise': ['2022'], 'STATION': ['CB4'], 'SMP_ID_PROVIDER': [1],
    'I129_at_kg ( x 10^7)': [6.4],  'unc_I129_at_kg ( x 10^7)': [0.2],
    'U238_ppb': [3.0], 'unc_U238_ppb': [0.1],
    'U236_at_kg': [15.0], 'unc_U236_at_kg': [1.5],  # no suffix: override applies
})}
MOCK_META = ['Cruise', 'STATION', 'SMP_ID_PROVIDER']
MOCK_OVERRIDES = {'U236_at_kg': 1e6, 'unc_U236_at_kg': 1e6}
tfm = Transformer(dfs_mock, cbs=[MeltNuclideColsCB(MOCK_META, JOIS_COL_LUT, UNIT_CONV_LUT, MOCK_OVERRIDES)])
tfm()
out = tfm.dfs['SEAWATER']
# I129: suffix factor 1e7 * base_factor 1.0
i129 = out[out['NUCLIDE'] == 'I129'].iloc[0]
test_eq(i129['VALUE'], 6.4 * 1e7);  test_eq(i129['UNC'], 0.2 * 1e7);  test_eq(i129['UNIT'], 'at_kg')
# U238: no suffix, base_factor = U238_PPB_TO_AT_KG, unit normalised to at_kg
u238 = out[out['NUCLIDE'] == 'U238'].iloc[0]
test_eq(u238['VALUE'], 3.0 * U238_PPB_TO_AT_KG);  test_eq(u238['UNIT'], 'at_kg')
# U236 at_kg: no suffix, override=1e6
u236 = out[(out['NUCLIDE'] == 'U236') & (out['UNIT'] == 'at_kg')].iloc[0]
test_eq(u236['VALUE'], 15.0 * 1e6);  test_eq(u236['UNC'], 1.5 * 1e6)
# Negative exponent support (future-proofing smoke test)
dfs_neg = {'SEAWATER': pd.DataFrame({
    'Cruise': ['X'], 'STATION': ['S'], 'SMP_ID_PROVIDER': [2],
    'I129_at_l ( x 10^-3)': [5.0], 'unc_I129_at_l ( x 10^-3)': [0.5],
})}
tfm2 = Transformer(dfs_neg, cbs=[MeltNuclideColsCB(MOCK_META, JOIS_COL_LUT, UNIT_CONV_LUT)])
tfm2()
row = tfm2.dfs['SEAWATER'].iloc[0]
test_eq(row['VALUE'], 5.0 * 10**-3)
print('MeltNuclideColsCB: suffix + unit_conv + override + negative exponent all correct. PASS')


:::{.callout-important}
## FEEDBACK TO DATA PROVIDER

Uncertainty values in 2024 are three orders of magnitude lower than in 2021-2023 (relative uncertainty ~0.0002% versus ~3-5%), with no documented change in analytical method or instrumentation. All three uncertainty representations (`unc_I129_at_kg`, `unc_I129_at_l`, `unc_129_pct`) are internally consistent within each year. The provider should confirm whether this reflects a genuine precision improvement or a data reporting issue.
:::


## Remap nomenclatures to MARIS identifiers

After `MeltAndScaleCB`, the DataFrame has string columns `NUCLIDE` (I129, U236, U238, U236_U238)
and `UNIT` (at_kg, at_l, at_ratio). U-238 values are already in atoms/kg; the ppb→atoms/kg
conversion was applied inside `MeltAndScaleCB` via `UNIT_CONV_LUT`. MARIS stores these as integer
foreign-key IDs from the central nomenclatures.

`RemapCB` maps source column values through a lookup table to a target column.
Named constants `LAB_ETH_LIP` and `AREA_BEAUFORT_SEA` replace the former magic numbers `345`/`4256`
and carry provenance comments pointing to their `get_lut()` verification query.


In [ ]:
#| exports
# MARIS nuclide IDs confirmed via get_lut('NUCLIDE')
NUCLIDE_LUT = {'I129': 28, 'U236': 108, 'U238': 64, 'U236_U238': 131}

# MARIS unit IDs confirmed via get_lut('UNIT')
UNIT_LUT = {'at_kg': 9, 'at_l': 12, 'at_ratio': 6}

# ETH Zurich / LIP (Casacuberta group) lab ID -- confirmed via get_lut('LABS')
LAB_ETH_LIP = 345

# Beaufort Sea area ID -- confirmed via get_lut('AREAS')
AREA_BEAUFORT_SEA = 4256


In [ ]:
# Verify RemapCB assigns correct NUCLIDE, UNIT, LAB, AREA IDs using named constants
dfs_mock = {'SEAWATER': pd.DataFrame({
    'NUCLIDE': ['I129', 'U236', 'U238'],
    'UNIT':    ['at_kg', 'at_l', 'at_kg'],
    'VALUE':   [1.0, 2.0, 3.0],
    'UNC':     [0.1, 0.2, 0.3],
})}
tfm = Transformer(dfs_mock, cbs=[
    RemapCB(lut=NUCLIDE_LUT, col_remap='NUCLIDE', col_src='NUCLIDE'),
    RemapCB(lut=UNIT_LUT,    col_remap='UNIT',    col_src='UNIT'),
    RemapCB(lut={}, col_remap='LAB',  col_src='NUCLIDE', default_val=LAB_ETH_LIP),
    RemapCB(lut={}, col_remap='AREA', col_src='NUCLIDE', default_val=AREA_BEAUFORT_SEA),
])
tfm()
out = tfm.dfs['SEAWATER']
test_eq(out['NUCLIDE'].tolist(), [28, 108, 64])
test_eq(out['UNIT'].tolist(),    [9,  12,  9])
test_eq(out['LAB'].tolist(),     [LAB_ETH_LIP]        * 3)
test_eq(out['AREA'].tolist(),    [AREA_BEAUFORT_SEA]   * 3)
print("RemapCB: all nomenclatures mapped to correct MARIS IDs. ✓")


RemapCB: all nomenclatures mapped to correct MARIS IDs. ✓


## Standardise final columns

Three shared callbacks complete the pipeline:

- `SanitizeLonLatCB`: validates longitude/latitude ranges and ensures correct sign convention
- `EncodeTimeCB`: encodes the TIME column into the NetCDF-compatible numeric representation
- `AddSampleIDCB`: assigns a sequential `SMP_ID` and preserves the provider's `SMP_ID_PROVIDER`

All three are imported from `marisco.callbacks` and require no configuration for JOIS.

In [ ]:
#|eval: false
tfm = Transformer(dfs, cbs=[
    RenameColsCB(), ParseDateTimeCB(),
    MeltNuclideColsCB(META_COLS, JOIS_COL_LUT, UNIT_CONV_LUT, COL_SCALE_OVERRIDES),
    RemapCB(lut=NUCLIDE_LUT, col_remap='NUCLIDE', col_src='NUCLIDE'),
    RemapCB(lut=UNIT_LUT,    col_remap='UNIT',    col_src='UNIT'),
    RemapCB(lut={}, col_remap='LAB',  col_src='NUCLIDE', default_val=LAB_ETH_LIP),
    RemapCB(lut={}, col_remap='AREA', col_src='NUCLIDE', default_val=AREA_BEAUFORT_SEA),
    SanitizeLonLatCB(),
    EncodeTimeCB(),
    AddSampleIDCB(col_provider='SMP_ID_PROVIDER'),
])
tfm()
out = tfm.dfs['SEAWATER']
print(f"Final shape: {out.shape}")
print("Columns:", out.columns.tolist())
print(out[['SMP_ID', 'SMP_ID_PROVIDER', 'NUCLIDE', 'UNIT', 'LAB', 'AREA']].head(4).to_string())


Final shape: (2016, 17)


Columns: ['Cruise', 'STATION', 'SMP_ID_PROVIDER', 'LAT', 'LON', 'TIME', 'Pressure_dbar', 'SMP_DEPTH', 'TEMP', 'SAL', 'VALUE', 'UNC', 'NUCLIDE', 'UNIT', 'LAB', 'AREA', 'SMP_ID']


   SMP_ID SMP_ID_PROVIDER  NUCLIDE  UNIT  LAB  AREA
0       1           289.0       28     9  345  4256
1       2           290.0       28     9  345  4256
2       3           291.0       28     9  345  4256
3       4           292.0       28     9  345  4256


In [ ]:
#|eval: false
print("Final data summary (uppercase columns only):")
upper_cols = [c for c in out.columns if c.isupper()]
print(out[upper_cols].describe().to_string())

Final data summary (uppercase columns only):


               LAT          LON          TIME    SMP_DEPTH         TEMP          SAL         VALUE           UNC      NUCLIDE         UNIT     LAB    AREA       SMP_ID
count  2016.000000  2016.000000  2.016000e+03  2016.000000  2016.000000  2016.000000  2.016000e+03  2.015000e+03  2016.000000  2016.000000  2016.0  2016.0  2016.000000
mean     74.860128  -145.420826  1.670466e+09   750.954670    -0.133983    33.437905  1.088666e+12  6.931932e+10    70.189980     9.650298   345.0  4256.0  1008.500000
std       2.590004     7.100110  2.049542e+07   893.294636     0.835686     2.459200  2.702033e+12  2.975590e+11    41.316164     2.018581     0.0     0.0   582.113391
min      70.539500  -153.320167  1.630157e+09     4.179505    -1.586100    24.756100  0.000000e+00  0.000000e+00    28.000000     6.000000   345.0  4256.0     1.000000
25%      72.599667  -151.575833  1.664062e+09   151.645681    -0.549900    33.105800  9.085735e+06  2.997537e+05    28.000000     9.000000   345.0  4256.0   504

## NetCDF encoder

The encoder wraps the full pipeline and writes the standardised data to a NetCDF4 file. Global attributes are assembled via `GlobAttrsFeeder` with `BboxCB`, `DepthRangeCB`, `TimeRangeCB`, plus keywords and processing logs.

We do not yet have an INIS entry for the JOIS datasets, so `INISCB` is commented out. The [IAEA INIS repository](https://www.iaea.org/resources/databases/inis) will be used for bibliographic metadata once the record is created. A placeholder line is included for future use.

In [ ]:
#| exports
# NetCDF global attributes
JOIS_KEYWORDS = ['Beaufort Sea', 'JOIS', 'I-129', 'U-236', 'U-238', 'radionuclides', 'seawater', 'Arctic']

def get_attrs(tfm):
    "Retrieve global attributes for the JOIS handler."
    return GlobAttrsFeeder(tfm.dfs, cbs=[
        BboxCB(),
        DepthRangeCB(),
        TimeRangeCB(),
        #INISCB('XXXXXXXX'),  # TODO: add INIS record id when available
        KeyValuePairCB('keywords', ', '.join(JOIS_KEYWORDS)),
        KeyValuePairCB('publisher_postprocess_logs', ', '.join(tfm.logs)),
    ])()

In [ ]:
#| exports
def encode(fname_out=None  # Output NetCDF file path; defaults to fname_out
            ):
    "Encode JOIS data to NetCDF4."
    fname_out = fname_out or globals().get('fname_out', 'JOIS_Beaufort_Sea.nc')
    dfs = load_data()
    tfm = Transformer(dfs, cbs=[
        RenameColsCB(), ParseDateTimeCB(),
        MeltNuclideColsCB(META_COLS, JOIS_COL_LUT, UNIT_CONV_LUT, COL_SCALE_OVERRIDES),
        RemapCB(lut=NUCLIDE_LUT, col_remap='NUCLIDE', col_src='NUCLIDE'),
        RemapCB(lut=UNIT_LUT,    col_remap='UNIT',    col_src='UNIT'),
        RemapCB(lut={}, col_remap='LAB',  col_src='NUCLIDE', default_val=LAB_ETH_LIP),
        RemapCB(lut={}, col_remap='AREA', col_src='NUCLIDE', default_val=AREA_BEAUFORT_SEA),
        SanitizeLonLatCB(),
        EncodeTimeCB(),
        AddSampleIDCB(col_provider='SMP_ID_PROVIDER'),
    ])
    tfm()
    encoder = NetCDFEncoder(tfm.dfs, dest_fname=fname_out,
                            global_attrs=get_attrs(tfm))
    encoder.encode()


In [ ]:
#|eval: false
# Encode to NetCDF
encode('../../_data/output/jois.nc')
print("JOIS NetCDF written.")

JOIS NetCDF written.
